# Neural Machine Translation: Mandarin Chinese → English
### Seq2Seq with a BiLSTM encoder, an LSTM decoder and multiplicative attention (CS224N Spring 2024, Assignment 3)

This notebook trains the assignment's NMT system end to end on a Kaggle GPU:

1. **Data**: downloads the official starter package (GALE Mandarin–English news, with 200k training pairs, 1k dev and 1k test) together with its SentencePiece models and `vocab.json`.
2. **Implementation**: writes the completed code, namely `pad_sents` (Q1a), `ModelEmbeddings` (Q1b) and `NMT.__init__ / encode / decode / step` (Q1c–f), plus small fixes so the starter code runs on current PyTorch and Kaggle images.
3. **Sanity checks**: runs the official `sanity_check.py 1d / 1e / 1f`.
4. **Training**: uses the course's GPU recipe (`sh run.sh train`).
5. **Evaluation**: corpus BLEU on the test set (beam size 10; the assignment expects more than 18), sample translations, and the beam-search diagnostics used in Q2(d).
6. **Packaging**: builds `assignment3.zip` with the same layout as `collect_submission.sh`.

**Before running:** open the notebook settings, set **Accelerator → GPU T4 x2**, and turn **Internet on** (it is needed to download the data and two pip packages). Then use **Save Version → Save & Run All (Commit)** so training keeps running in the background. Training usually takes 1.5–3 hours, and the run stops itself well inside Kaggle's 12-hour session limit.

## Model

| Stage | Computation |
|---|---|
| Embeddings | $\mathbf{x}_i \in \mathbb{R}^{e}$ for every source subword ($e = 1024$) |
| Conv1d | kernel size 2 with `padding='same'`: each subword is mixed with its right neighbour, which helps compose multi-character Chinese words (Q2a) |
| Encoder | bidirectional LSTM giving $\mathbf{h}^{enc}_i \in \mathbb{R}^{2h}$ and $\mathbf{c}^{enc}_i$ ($h = 768$) |
| Decoder init | $\mathbf{h}^{dec}_0 = \mathbf{W}_h[\overleftarrow{\mathbf{h}}^{enc}_1; \overrightarrow{\mathbf{h}}^{enc}_m]$, $\;\mathbf{c}^{dec}_0 = \mathbf{W}_c[\overleftarrow{\mathbf{c}}^{enc}_1; \overrightarrow{\mathbf{c}}^{enc}_m]$ |
| Decoder step | LSTMCell on $\bar{\mathbf{y}}_t = [\mathbf{y}_t; \mathbf{o}_{t-1}]$ (input feeding) |
| Attention | $e_{t,i} = (\mathbf{h}^{dec}_t)^\top \mathbf{W}_{attProj}\,\mathbf{h}^{enc}_i$, $\;\alpha_t = \mathrm{softmax}(e_t)$ with padding masked to $-\infty$, $\;\mathbf{a}_t = \sum_i \alpha_{t,i}\mathbf{h}^{enc}_i$ |
| Output | $\mathbf{o}_t = \mathrm{dropout}(\tanh(\mathbf{W}_u[\mathbf{a}_t; \mathbf{h}^{dec}_t]))$, $\;P_t = \mathrm{softmax}(\mathbf{W}_{vocab}\,\mathbf{o}_t)$ |

The model has 62.1M parameters. Training minimises the cross-entropy of the gold target subwords (teacher forcing) using Adam (lr 5e-4), gradient clipping at 5, dropout 0.3 and batches of 32. Every 200 iterations the dev perplexity is checked. If it improved, the checkpoint is saved; if not, the best checkpoint is reloaded and the learning rate is halved. Training stops after the fifth such decay.

## 1. Configuration

In [ ]:
import os, re, sys, json, time, shutil, hashlib, zipfile, itertools, subprocess, urllib.request
from pathlib import Path

# Set QUICK_RUN = True (or export NMT_QUICK_RUN=1) for a ~5-minute end-to-end dry run that uses a
# tiny model and a small data subset. Keep it False for the real training run.
QUICK_RUN = os.environ.get("NMT_QUICK_RUN", "0") == "1"

ON_KAGGLE = Path("/kaggle/working").is_dir()
WORK_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "nmt_work"
CODE_DIR = WORK_DIR / "student"       # assignment code; checkpoints, runs/ and outputs/ land here
# The corpus stays outside /kaggle/working so it is not republished with the notebook output (LDC licence).
DATA_ROOT = Path("/tmp/nmt_data") if ON_KAGGLE else WORK_DIR / "data"
DATA_DIR = DATA_ROOT / "zh_en_data"
LOG_DIR = WORK_DIR / "logs"

STARTER_URL = "https://web.stanford.edu/class/archive/cs/cs224n/cs224n.1246/assignments/a3_spr24_student_code.zip"
STARTER_SHA256 = "7a309d1eb55d881c06b2860b6b7d6551faa97c3b6cb6c52063324e3278f851cd"

# Course settings: `run.sh train` flags + run.py defaults (embed 1024, hidden 768, beam size 10)
TRAIN_ARGS = ["--lr=5e-4", "--patience=1", "--valid-niter=200", "--batch-size=32", "--dropout=.3"]
VALID_NITER = 200
SHOW_EVERY = 1000              # echo the training-progress line every N iterations (train.log has all of them)
TRAIN_TIME_BUDGET_H = 10.5     # stop training (keeping the best checkpoint) to stay inside Kaggle's 12 h limit
KEEP_OPTIMIZER_STATE = False   # model.bin.optim is ~500 MB and only needed to resume training

if QUICK_RUN:
    TRAIN_ARGS = ["--lr=5e-4", "--patience=1", "--valid-niter=20", "--batch-size=32", "--dropout=.3",
                  "--embed-size=64", "--hidden-size=64", "--max-epoch=2", "--log-every=5"]
    VALID_NITER, SHOW_EVERY = 20, 20
    SUBSET = {"train": 2000, "dev": 100, "test": 50}

for d in (CODE_DIR, DATA_ROOT, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"QUICK_RUN={QUICK_RUN}\ncode: {CODE_DIR}\ndata: {DATA_DIR}")

In [ ]:
def run(cmd, log_path=None, show=lambda line: True, deadline=None, can_stop=lambda line: True):
    """Run `cmd` inside CODE_DIR, echo the lines `show` accepts and tee every line to `log_path`.
    After `deadline` (a time.time() value) the process is terminated at the next line where
    `can_stop(line)` is true."""
    cmd = [str(c) for c in cmd]
    print("$", "python" if cmd[0] == sys.executable else cmd[0], *cmd[1:], flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONWARNINGS": "ignore", "PYTHONIOENCODING": "utf-8"}
    lines, stopped = [], False
    with open(log_path or os.devnull, "w", encoding="utf-8") as log, \
         subprocess.Popen(cmd, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as proc:
        for line in proc.stdout:
            lines.append(line)
            log.write(line)
            if line.strip() and show(line):
                print(line, end="", flush=True)
            if deadline is not None and time.time() > deadline and can_stop(line):
                stopped = True
                proc.terminate()
                break
        proc.wait()
    if stopped:
        print("[time budget reached: training stopped, continuing with the best checkpoint saved so far]")
    elif proc.returncode != 0:
        print("".join(lines[-30:]))
        raise RuntimeError(f"{' '.join(cmd[1:3])} exited with code {proc.returncode}")
    return "".join(lines)

## 2. Environment check

In [ ]:
import torch

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")
USE_CUDA = torch.cuda.is_available()
if USE_CUDA:
    GPU_NAME = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    print(f"GPU: {GPU_NAME} (compute capability {major}.{minor}), {torch.cuda.device_count()} device(s)")
    try:  # recent PyTorch wheels dropped Pascal kernels (P100), so fail fast with a clear message
        x = torch.randn(256, 256, device="cuda")
        (x @ x).sum().item()
    except Exception as err:
        raise RuntimeError(f"This PyTorch build cannot run on {GPU_NAME}. "
                           "Switch Settings → Accelerator to 'GPU T4 x2' and run again.") from err
elif QUICK_RUN:
    GPU_NAME = "cpu"
    print("No GPU found. That is fine for QUICK_RUN, which runs on the CPU.")
else:
    raise RuntimeError("No GPU found. Enable Settings → Accelerator → 'GPU T4 x2'. On a CPU each "
                       "iteration takes ~3 s, and a full run needs ~20k iterations.")
CUDA_FLAG = ["--cuda"] if USE_CUDA else []

In [ ]:
import importlib.util

REQUIRED = {"docopt": "docopt", "sacrebleu": "sacrebleu", "sentencepiece": "sentencepiece",
            "nltk": "nltk", "tensorboard": "tensorboard", "tqdm": "tqdm"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", *missing)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    except subprocess.CalledProcessError as err:
        raise RuntimeError("pip install failed. Turn on Settings → Internet and run again.") from err
print("all requirements available")

## 3. Data and starter code

The official starter package provides the corpus, the SentencePiece models (`src.model` with 21k pieces for Chinese, `tgt.model` with 8k pieces for English), `vocab.json`, the sanity-check fixtures and the unchanged starter files (`vocab.py`, `sanity_check.py`). To avoid the download, attach a Kaggle dataset that contains `a3_spr24_student_code.zip`.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

attached = sorted(Path("/kaggle/input").glob("**/a3_spr24_student_code.zip")) if ON_KAGGLE else []
zip_path = attached[0] if attached else DATA_ROOT / "a3_spr24_student_code.zip"
if not zip_path.exists():
    print("downloading", STARTER_URL)
    try:
        urllib.request.urlretrieve(STARTER_URL, zip_path)
    except OSError as err:
        raise RuntimeError("Download failed. Turn on Settings → Internet, or attach a dataset "
                           "containing a3_spr24_student_code.zip.") from err
digest = sha256(zip_path)
if digest != STARTER_SHA256:
    print(f"WARNING: sha256 {digest} differs from the starter version this notebook was tested with")

with zipfile.ZipFile(zip_path) as zf:
    for member in zf.namelist():
        rel = Path(member)
        if member.endswith("/") or rel.parts[0] != "student" or ".." in rel.parts:
            continue
        rel = rel.relative_to("student")
        dest = (DATA_ROOT if rel.parts[0] == "zh_en_data" else CODE_DIR) / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(member) as src, open(dest, "wb") as out:
            shutil.copyfileobj(src, out)

if QUICK_RUN:  # small aligned subset, cut with the same line splitting that read_corpus uses
    subset_dir = DATA_ROOT / "zh_en_subset"
    subset_dir.mkdir(exist_ok=True)
    for split, n in SUBSET.items():
        for lang in ("zh", "en"):
            with open(DATA_DIR / f"{split}.{lang}", encoding="utf-8") as src, \
                 open(subset_dir / f"{split}.{lang}", "w", encoding="utf-8") as out:
                out.writelines(itertools.islice(src, n))
    DATA_DIR = subset_dir

for split in ("train", "dev", "test"):
    with open(DATA_DIR / f"{split}.zh", encoding="utf-8") as f_zh, open(DATA_DIR / f"{split}.en", encoding="utf-8") as f_en:
        n_zh, n_en = sum(1 for _ in f_zh), sum(1 for _ in f_en)
    assert n_zh == n_en, (split, n_zh, n_en)
    print(f"{split:>5}: {n_zh:,} sentence pairs")
with open(DATA_DIR / "dev.zh", encoding="utf-8") as f_zh, open(DATA_DIR / "dev.en", encoding="utf-8") as f_en:
    zh, en = next(itertools.islice(f_zh, 3, None)).strip(), next(itertools.islice(f_en, 3, None)).strip()
print(f"\ndev[3], the example traced by the beam-search diagnostics:\n  {zh}\n  {en}")

os.chdir(CODE_DIR)  # the %%writefile cells below write into CODE_DIR

## 4. Implementation

The next cells overwrite the starter files with their completed versions. The solutions are inside the `### YOUR CODE HERE` blocks:

- `utils.py`, **Q1(a)** `pad_sents`: right-pads every sentence to the length of the longest one in the batch.
- `model_embeddings.py`, **Q1(b)**: source and target `nn.Embedding` layers with their `<pad>` indices.
- `nmt_model.py`, **Q1(c)**: creates the layers in the required order. **Q1(d)** `encode`: embeddings, then a Conv1d over time, then a packed BiLSTM, then the projected decoder initial state. **Q1(e)** `decode`: a teacher-forced loop over target steps with input feeding. **Q1(f)** `step`: LSTMCell, attention scores via `bmm`, masked softmax, context vector and combined output.

Compatibility fixes outside the graded blocks, needed on current Kaggle images:

- `torch.load(..., weights_only=False)` for checkpoints, because they pickle the `Vocab` and PyTorch 2.6+ defaults to `weights_only=True`.
- `beam_search_diagnostics.py`: `os.getlogin()` raises inside containers, so it falls back to `getpass.getuser()`. The JSON is also written as UTF-8.
- `utils.py`: also downloads NLTK's `punkt_tab`, which `word_tokenize` requires from NLTK 3.9 onwards.
- `run.py`: the course's hard-coded sizes (embed 1024, hidden 768, beam 10) are now the CLI defaults, so the same script can also run the quick dry run.

PyTorch's bidirectional LSTM concatenates `[forward; backward]` rather than the handout's `[backward; forward]`. This is a fixed permutation that the learned projections absorb.

In [ ]:
%%writefile utils.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
CS224N 2022-23: Homework 4
utils.py: Utility Functions
Pencheng Yin <pcyin@cs.cmu.edu>
Sahil Chopra <schopra8@stanford.edu>
Vera Lin <veralin@stanford.edu>
Siyan Li <siyanli@stanford.edu>
Moussa KB Doumbouya <moussa@stanford.edu>
"""

from typing import List
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import nltk
import sentencepiece as spm
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # nltk >= 3.9 tokenizes with punkt_tab instead of punkt


def pad_sents(sents, pad_token):
    """ Pad list of sentences according to the longest sentence in the batch.
        The paddings should be at the end of each sentence.
    @param sents (list[list[str]]): list of sentences, where each sentence
                                    is represented as a list of words
    @param pad_token (str): padding token
    @returns sents_padded (list[list[str]]): list of sentences where sentences shorter
        than the max length sentence are padded out with the pad_token, such that
        each sentences in the batch now has equal length.
    """
    sents_padded = []

    ### YOUR CODE HERE (~6 Lines)
    max_len = max(len(s) for s in sents)
    for s in sents:
        sents_padded.append(s + [pad_token] * (max_len - len(s)))
    ### END YOUR CODE

    return sents_padded


def read_corpus(file_path, source, vocab_size=2500):
    """ Read file, where each sentence is dilineated by a `\n`.
    @param file_path (str): path to file containing corpus
    @param source (str): "tgt" or "src" indicating whether text
        is of the source language or target language
    @param vocab_size (int): number of unique subwords in
        vocabulary when reading and tokenizing
    """
    data = []
    sp = spm.SentencePieceProcessor()
    sp.load('{}.model'.format(source))

    with open(file_path, 'r', encoding='utf8') as f:
        for line in f:
            subword_tokens = sp.encode_as_pieces(line)
            # only append <s> and </s> to the target sentence
            if source == 'tgt':
                subword_tokens = ['<s>'] + subword_tokens + ['</s>']
            data.append(subword_tokens)

    return data


def autograder_read_corpus(file_path, source):
    """ Read file, where each sentence is dilineated by a `\n`.
    @param file_path (str): path to file containing corpus
    @param source (str): "tgt" or "src" indicating whether text
        is of the source language or target language
    """
    data = []
    for line in open(file_path, encoding='utf8'):
        sent = nltk.word_tokenize(line)
        # only append <s> and </s> to the target sentence
        if source == 'tgt':
            sent = ['<s>'] + sent + ['</s>']
        data.append(sent)

    return data


def batch_iter(data, batch_size, shuffle=False):
    """ Yield batches of source and target sentences reverse sorted by length (largest to smallest).
    @param data (list of (src_sent, tgt_sent)): list of tuples containing source and target sentence
    @param batch_size (int): batch size
    @param shuffle (boolean): whether to randomly shuffle the dataset
    """
    batch_num = math.ceil(len(data) / batch_size)
    index_array = list(range(len(data)))

    if shuffle:
        np.random.shuffle(index_array)

    for i in range(batch_num):
        indices = index_array[i * batch_size: (i + 1) * batch_size]
        examples = [data[idx] for idx in indices]

        examples = sorted(examples, key=lambda e: len(e[0]), reverse=True)
        src_sents = [e[0] for e in examples]
        tgt_sents = [e[1] for e in examples]

        yield src_sents, tgt_sents

In [ ]:
%%writefile model_embeddings.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
CS224N Spring 2024: Assignment 3
model_embeddings.py: Embeddings for the NMT model
Pencheng Yin <pcyin@cs.cmu.edu>
Sahil Chopra <schopra8@stanford.edu>
Anand Dhoot <anandd@stanford.edu>
Vera Lin <veralin@stanford.edu>
Siyan Li <siyanli@stanford.edu>
"""

import torch.nn as nn

class ModelEmbeddings(nn.Module): 
    """
    Class that converts input words to their embeddings.
    """
    def __init__(self, embed_size, vocab):
        """
        Init the Embedding layers.

        @param embed_size (int): Embedding size (dimensionality)
        @param vocab (Vocab): Vocabulary object containing src and tgt languages
                              See vocab.py for documentation.
        """
        super(ModelEmbeddings, self).__init__()
        self.embed_size = embed_size

        # default values
        self.source = None
        self.target = None

        src_pad_token_idx = vocab.src['<pad>']
        tgt_pad_token_idx = vocab.tgt['<pad>']

        ### YOUR CODE HERE (~2 Lines)
        ### TODO - Initialize the following variables:
        ###     self.source (Embedding Layer for source language)
        ###     self.target (Embedding Layer for target langauge)
        ###
        ### Note:
        ###     1. `vocab` object contains two vocabularies:
        ###            `vocab.src` for source
        ###            `vocab.tgt` for target
        ###     2. You can get the length of a specific vocabulary by running:
        ###             `len(vocab.<specific_vocabulary>)`
        ###     3. Remember to include the padding token for the specific vocabulary
        ###        when creating your Embedding.
        ###
        ### Use the following docs to properly initialize these variables:
        ###     Embedding Layer:
        ###         https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html
        self.source = nn.Embedding(len(vocab.src), embed_size, padding_idx=src_pad_token_idx)
        self.target = nn.Embedding(len(vocab.tgt), embed_size, padding_idx=tgt_pad_token_idx)
        ### END YOUR CODE

In [ ]:
%%writefile nmt_model.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
CS224N Spring 2024: Homework 3
nmt_model.py: NMT Model
Pencheng Yin <pcyin@cs.cmu.edu>
Sahil Chopra <schopra8@stanford.edu>
Vera Lin <veralin@stanford.edu>
Siyan Li <siyanli@stanford.edu>
Moussa Doumbouya <moussa@stanford.edu>
"""

from collections import namedtuple
import sys
from typing import List, Tuple, Dict, Set, Union
import torch
import torch.nn as nn
import torch.nn.utils
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_packed_sequence, pack_padded_sequence

from model_embeddings import ModelEmbeddings

Hypothesis = namedtuple('Hypothesis', ['value', 'score'])


class NMT(nn.Module):
    """ Simple Neural Machine Translation Model:
        - Bidrectional LSTM Encoder
        - Unidirection LSTM Decoder
        - Global Attention Model (Luong, et al. 2015)
    """

    def __init__(self, embed_size, hidden_size, vocab, dropout_rate=0.2):
        """ Init NMT Model.

        @param embed_size (int): Embedding size (dimensionality)
        @param hidden_size (int): Hidden Size, the size of hidden states (dimensionality)
        @param vocab (Vocab): Vocabulary object containing src and tgt languages
                              See vocab.py for documentation.
        @param dropout_rate (float): Dropout probability, for attention
        """
        super(NMT, self).__init__()
        self.model_embeddings = ModelEmbeddings(embed_size, vocab)
        self.hidden_size = hidden_size
        self.dropout_rate = dropout_rate
        self.vocab = vocab

        # default values
        self.post_embed_cnn = None
        self.encoder = None
        self.decoder = None
        self.h_projection = None
        self.c_projection = None
        self.att_projection = None
        self.combined_output_projection = None
        self.target_vocab_projection = None
        self.dropout = None
        # For sanity check only, not relevant to implementation
        self.gen_sanity_check = False
        self.counter = 0

        ### YOUR CODE HERE (~9 Lines)
        ### TODO - Initialize the following variables IN THIS ORDER:
        ###     self.post_embed_cnn (Conv1d layer with kernel size 2, input and output channels = embed_size,
        ###         padding = same to preserve output shape )
        ###     self.encoder (Bidirectional LSTM with bias)
        ###     self.decoder (LSTM Cell with bias)
        ###     self.h_projection (Linear Layer with no bias), called W_{h} in the PDF.
        ###     self.c_projection (Linear Layer with no bias), called W_{c} in the PDF.
        ###     self.att_projection (Linear Layer with no bias), called W_{attProj} in the PDF.
        ###     self.combined_output_projection (Linear Layer with no bias), called W_{u} in the PDF.
        ###     self.target_vocab_projection (Linear Layer with no bias), called W_{vocab} in the PDF.
        ###     self.dropout (Dropout Layer)
        ###
        ### Use the following docs to properly initialize these variables:
        ###     Conv1d:
        ###         https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html
        ###     LSTM:
        ###         https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
        ###     LSTM Cell:
        ###         https://pytorch.org/docs/stable/generated/torch.nn.LSTMCell.html
        ###     Linear Layer:
        ###         https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
        ###     Dropout Layer:
        ###         https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html
        self.post_embed_cnn = nn.Conv1d(in_channels=embed_size, out_channels=embed_size, kernel_size=2,
                                        padding='same')
        self.encoder = nn.LSTM(input_size=embed_size, hidden_size=hidden_size, bias=True, bidirectional=True)
        self.decoder = nn.LSTMCell(input_size=embed_size + hidden_size, hidden_size=hidden_size, bias=True)
        self.h_projection = nn.Linear(2 * hidden_size, hidden_size, bias=False)
        self.c_projection = nn.Linear(2 * hidden_size, hidden_size, bias=False)
        self.att_projection = nn.Linear(2 * hidden_size, hidden_size, bias=False)
        self.combined_output_projection = nn.Linear(3 * hidden_size, hidden_size, bias=False)
        self.target_vocab_projection = nn.Linear(hidden_size, len(vocab.tgt), bias=False)
        self.dropout = nn.Dropout(p=dropout_rate)
        ### END YOUR CODE

    def forward(self, source: List[List[str]], target: List[List[str]]) -> torch.Tensor:
        """ Take a mini-batch of source and target sentences, compute the log-likelihood of
        target sentences under the language models learned by the NMT system.

        @param source (List[List[str]]): list of source sentence tokens
        @param target (List[List[str]]): list of target sentence tokens, wrapped by `<s>` and `</s>`

        @returns scores (Tensor): a variable/tensor of shape (b, ) representing the
                                    log-likelihood of generating the gold-standard target sentence for
                                    each example in the input batch. Here b = batch size.
        """
        # Compute sentence lengths
        source_lengths = [len(s) for s in source]

        # Convert list of lists into tensors
        source_padded = self.vocab.src.to_input_tensor(source, device=self.device)  # Tensor: (src_len, b)
        target_padded = self.vocab.tgt.to_input_tensor(target, device=self.device)  # Tensor: (tgt_len, b)

        ###     Run the network forward:
        ###     1. Apply the encoder to `source_padded` by calling `self.encode()`
        ###     2. Generate sentence masks for `source_padded` by calling `self.generate_sent_masks()`
        ###     3. Apply the decoder to compute combined-output by calling `self.decode()`
        ###     4. Compute log probability distribution over the target vocabulary using the
        ###        combined_outputs returned by the `self.decode()` function.

        enc_hiddens, dec_init_state = self.encode(source_padded, source_lengths)
        enc_masks = self.generate_sent_masks(enc_hiddens, source_lengths)
        combined_outputs = self.decode(enc_hiddens, enc_masks, dec_init_state, target_padded)
        P = F.log_softmax(self.target_vocab_projection(combined_outputs), dim=-1)

        # Zero out, probabilities for which we have nothing in the target text
        target_masks = (target_padded != self.vocab.tgt['<pad>']).float()

        # Compute log probability of generating true target words
        target_gold_words_log_prob = torch.gather(P, index=target_padded[1:].unsqueeze(-1), dim=-1).squeeze(
            -1) * target_masks[1:]
        scores = target_gold_words_log_prob.sum(dim=0)
        return scores

    def encode(self, source_padded: torch.Tensor, source_lengths: List[int]) -> Tuple[
        torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        """ Apply the encoder to source sentences to obtain encoder hidden states.
            Additionally, take the final states of the encoder and project them to obtain initial states for decoder.

        @param source_padded (Tensor): Tensor of padded source sentences with shape (src_len, b), where
                                        b = batch_size, src_len = maximum source sentence length. Note that
                                       these have already been sorted in order of longest to shortest sentence.
        @param source_lengths (List[int]): List of actual lengths for each of the source sentences in the batch
        @returns enc_hiddens (Tensor): Tensor of hidden units with shape (b, src_len, h*2), where
                                        b = batch size, src_len = maximum source sentence length, h = hidden size.
        @returns dec_init_state (tuple(Tensor, Tensor)): Tuple of tensors representing the decoder's initial
                                                hidden state and cell.
        """
        enc_hiddens, dec_init_state = None, None

        ### YOUR CODE HERE (~ 11 Lines)
        ### TODO:
        ###     1. Construct Tensor `X` of source sentences with shape (src_len, b, e) using the source model embeddings.
        ###         src_len = maximum source sentence length, b = batch size, e = embedding size. Note
        ###         that there is no initial hidden state or cell for the encoder.
        ###     2. Apply the post_embed_cnn layer. Before feeding X into the CNN, first use torch.permute to change the
        ###         shape of X to (b, e, src_len). After getting the output from the CNN, remember to use torch.permute
        ###         again to revert X back to its original shape.
        ###     3. Compute `enc_hiddens`, `last_hidden`, `last_cell` by applying the encoder to `X`.
        ###         - Before you can apply the encoder, you need to apply the `pack_padded_sequence` function to X.
        ###         - After you apply the encoder, you need to apply the `pad_packed_sequence` function to enc_hiddens.
        ###         - Note that the shape of the tensor returned by the encoder is (src_len, b, h*2) and we want to
        ###           return a tensor of shape (b, src_len, h*2) as `enc_hiddens`.
        ###     4. Compute `dec_init_state` = (init_decoder_hidden, init_decoder_cell):
        ###         - `init_decoder_hidden`:
        ###             `last_hidden` is a tensor shape (2, b, h). The first dimension corresponds to forwards and backwards.
        ###             Concatenate the forwards and backwards tensors to obtain a tensor shape (b, 2*h).
        ###             Apply the h_projection layer to this in order to compute init_decoder_hidden.
        ###             This is h_0^{dec} in the PDF. Here b = batch size, h = hidden size
        ###         - `init_decoder_cell`:
        ###             `last_cell` is a tensor shape (2, b, h). The first dimension corresponds to forwards and backwards.
        ###             Concatenate the forwards and backwards tensors to obtain a tensor shape (b, 2*h).
        ###             Apply the c_projection layer to this in order to compute init_decoder_cell.
        ###             This is c_0^{dec} in the PDF. Here b = batch size, h = hidden size
        ###
        ### See the following docs, as you may need to use some of the following functions in your implementation:
        ###     Pack the padded sequence X before passing to the encoder:
        ###         https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html
        ###     Pad the packed sequence, enc_hiddens, returned by the encoder:
        ###         https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_packed_sequence.html
        ###     Tensor Concatenation:
        ###         https://pytorch.org/docs/stable/generated/torch.cat.html
        ###     Tensor Permute:
        ###         https://pytorch.org/docs/stable/generated/torch.permute.html
        X = self.model_embeddings.source(source_padded)  # (src_len, b, e)
        X = self.post_embed_cnn(X.permute(1, 2, 0)).permute(2, 0, 1)  # conv over time: (b, e, src_len) -> (src_len, b, e)
        enc_hiddens, (last_hidden, last_cell) = self.encoder(pack_padded_sequence(X, source_lengths))
        enc_hiddens, _ = pad_packed_sequence(enc_hiddens, batch_first=True)  # (b, src_len, 2h)
        init_decoder_hidden = self.h_projection(torch.cat((last_hidden[0], last_hidden[1]), dim=1))  # (b, h)
        init_decoder_cell = self.c_projection(torch.cat((last_cell[0], last_cell[1]), dim=1))  # (b, h)
        dec_init_state = (init_decoder_hidden, init_decoder_cell)
        ### END YOUR CODE

        return enc_hiddens, dec_init_state

    def decode(self, enc_hiddens: torch.Tensor, enc_masks: torch.Tensor,
               dec_init_state: Tuple[torch.Tensor, torch.Tensor], target_padded: torch.Tensor) -> torch.Tensor:
        """Compute combined output vectors for a batch.

        @param enc_hiddens (Tensor): Hidden states (b, src_len, h*2), where
                                     b = batch size, src_len = maximum source sentence length, h = hidden size.
        @param enc_masks (Tensor): Tensor of sentence masks (b, src_len), where
                                     b = batch size, src_len = maximum source sentence length.
        @param dec_init_state (tuple(Tensor, Tensor)): Initial state and cell for decoder
        @param target_padded (Tensor): Gold-standard padded target sentences (tgt_len, b), where
                                       tgt_len = maximum target sentence length, b = batch size.

        @returns combined_outputs (Tensor): combined output tensor  (tgt_len, b,  h), where
                                        tgt_len = maximum target sentence length, b = batch_size,  h = hidden size
        """
        # Chop off the <END> token for max length sentences.
        target_padded = target_padded[:-1]

        # Initialize the decoder state (hidden and cell)
        dec_state = dec_init_state

        # Initialize previous combined output vector o_{t-1} as zero
        batch_size = enc_hiddens.size(0)
        o_prev = torch.zeros(batch_size, self.hidden_size, device=self.device)

        # Initialize a list we will use to collect the combined output o_t on each step
        combined_outputs = []

        ### YOUR CODE HERE (~9 Lines)
        ### TODO:
        ###     1. Apply the attention projection layer to `enc_hiddens` to obtain `enc_hiddens_proj`,
        ###         which should be shape (b, src_len, h),
        ###         where b = batch size, src_len = maximum source length, h = hidden size.
        ###         This is applying W_{attProj} to h^enc, as described in the PDF.
        ###     2. Construct tensor `Y` of target sentences with shape (tgt_len, b, e) using the target model embeddings.
        ###         where tgt_len = maximum target sentence length, b = batch size, e = embedding size.
        ###     3. Use the torch.split function to iterate over the time dimension of Y.
        ###         Within the loop, this will give you Y_t of shape (1, b, e) where b = batch size, e = embedding size.
        ###             - Squeeze Y_t into a tensor of dimension (b, e).
        ###             - Construct Ybar_t by concatenating Y_t with o_prev on their last dimension
        ###             - Use the step function to compute the the Decoder's next (cell, state) values
        ###               as well as the new combined output o_t.
        ###             - Append o_t to combined_outputs
        ###             - Update o_prev to the new o_t.
        ###     4. Use torch.stack to convert combined_outputs from a list length tgt_len of
        ###         tensors shape (b, h), to a single tensor shape (tgt_len, b, h)
        ###         where tgt_len = maximum target sentence length, b = batch size, h = hidden size.
        ###
        ### Note:
        ###    - When using the squeeze() function make sure to specify the dimension you want to squeeze
        ###      over. Otherwise, you will remove the batch dimension accidentally, if batch_size = 1.
        ###
        ### You may find some of these functions useful:
        ###     Zeros Tensor:
        ###         https://pytorch.org/docs/stable/generated/torch.zeros.html
        ###     Tensor Splitting (iteration):
        ###         https://pytorch.org/docs/stable/generated/torch.split.html
        ###     Tensor Dimension Squeezing:
        ###         https://pytorch.org/docs/stable/generated/torch.squeeze.html
        ###     Tensor Concatenation:
        ###         https://pytorch.org/docs/stable/generated/torch.cat.html
        ###     Tensor Stacking:
        ###         https://pytorch.org/docs/stable/generated/torch.stack.html
        enc_hiddens_proj = self.att_projection(enc_hiddens)  # (b, src_len, h)
        Y = self.model_embeddings.target(target_padded)  # (tgt_len, b, e)
        for Y_t in torch.split(Y, 1, dim=0):
            Ybar_t = torch.cat((Y_t.squeeze(0), o_prev), dim=1)  # (b, e + h)
            dec_state, o_t, _ = self.step(Ybar_t, dec_state, enc_hiddens, enc_hiddens_proj, enc_masks)
            combined_outputs.append(o_t)
            o_prev = o_t
        combined_outputs = torch.stack(combined_outputs, dim=0)  # (tgt_len, b, h)
        ### END YOUR CODE

        return combined_outputs

    def step(self, Ybar_t: torch.Tensor,
             dec_state: Tuple[torch.Tensor, torch.Tensor],
             enc_hiddens: torch.Tensor,
             enc_hiddens_proj: torch.Tensor,
             enc_masks: torch.Tensor) -> Tuple[Tuple, torch.Tensor, torch.Tensor]:
        """ Compute one forward step of the LSTM decoder, including the attention computation.

        @param Ybar_t (Tensor): Concatenated Tensor of [Y_t o_prev], with shape (b, e + h). The input for the decoder,
                                where b = batch size, e = embedding size, h = hidden size.
        @param dec_state (tuple(Tensor, Tensor)): Tuple of tensors both with shape (b, h), where b = batch size, h = hidden size.
                First tensor is decoder's prev hidden state, second tensor is decoder's prev cell.
        @param enc_hiddens (Tensor): Encoder hidden states Tensor, with shape (b, src_len, h * 2), where b = batch size,
                                    src_len = maximum source length, h = hidden size.
        @param enc_hiddens_proj (Tensor): Encoder hidden states Tensor, projected from (h * 2) to h. Tensor is with shape (b, src_len, h),
                                    where b = batch size, src_len = maximum source length, h = hidden size.
        @param enc_masks (Tensor): Tensor of sentence masks shape (b, src_len),
                                    where b = batch size, src_len is maximum source length.

        @returns dec_state (tuple (Tensor, Tensor)): Tuple of tensors both shape (b, h), where b = batch size, h = hidden size.
                First tensor is decoder's new hidden state, second tensor is decoder's new cell.
        @returns combined_output (Tensor): Combined output Tensor at timestep t, shape (b, h), where b = batch size, h = hidden size.
        @returns e_t (Tensor): Tensor of shape (b, src_len). It is attention scores distribution.
                                Note: You will not use this outside of this function.
                                      We are simply returning this value so that we can sanity check
                                      your implementation.
        """

        combined_output = None

        ### YOUR CODE HERE (~3 Lines)
        ### TODO:
        ###     1. Apply the decoder to `Ybar_t` and `dec_state`to obtain the new dec_state.
        ###     2. Split dec_state into its two parts (dec_hidden, dec_cell)
        ###     3. Compute the attention scores e_t, a Tensor shape (b, src_len).
        ###        Note: b = batch_size, src_len = maximum source length, h = hidden size.
        ###
        ###       Hints:
        ###         - dec_hidden is shape (b, h) and corresponds to h^dec_t in the PDF (batched)
        ###         - enc_hiddens_proj is shape (b, src_len, h) and corresponds to W_{attProj} h^enc (batched).
        ###         - Use batched matrix multiplication (torch.bmm) to compute e_t (be careful about the input/ output shapes!)
        ###         - To get the tensors into the right shapes for bmm, you will need to do some squeezing and unsqueezing.
        ###         - When using the squeeze() function make sure to specify the dimension you want to squeeze
        ###             over. Otherwise, you will remove the batch dimension accidentally, if batch_size = 1.
        ###
        ### Use the following docs to implement this functionality:
        ###     Batch Multiplication:
        ###         https://pytorch.org/docs/stable/generated/torch.bmm.html
        ###     Tensor Unsqueeze:
        ###         https://pytorch.org/docs/stable/generated/torch.unsqueeze.html
        ###     Tensor Squeeze:
        ###         https://pytorch.org/docs/stable/generated/torch.squeeze.html
        dec_state = self.decoder(Ybar_t, dec_state)
        dec_hidden, dec_cell = dec_state
        e_t = torch.bmm(enc_hiddens_proj, dec_hidden.unsqueeze(2)).squeeze(2)  # (b, src_len)
        ### END YOUR CODE

        # Set e_t to -inf where enc_masks has 1
        if enc_masks is not None:
            e_t.data.masked_fill_(enc_masks.bool(), -float('inf'))

        ### YOUR CODE HERE (~6 Lines)
        ### TODO:
        ###     1. Apply softmax to e_t to yield alpha_t
        ###     2. Use batched matrix multiplication between alpha_t and enc_hiddens to obtain the
        ###         attention output vector, a_t.
        ###           - alpha_t is shape (b, src_len)
        ###           - enc_hiddens is shape (b, src_len, 2h)
        ###           - a_t should be shape (b, 2h)
        ###           - You will need to do some squeezing and unsqueezing.
        ###     Note: b = batch size, src_len = maximum source length, h = hidden size.
        ###
        ###     3. Concatenate dec_hidden with a_t to compute tensor U_t
        ###     4. Apply the combined output projection layer to U_t to compute tensor V_t
        ###     5. Compute tensor O_t by first applying the Tanh function and then the dropout layer.
        ###
        ### Use the following docs to implement this functionality:
        ###     Softmax:
        ###         https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html
        ###     Batch Multiplication:
        ###         https://pytorch.org/docs/stable/generated/torch.bmm.html
        ###     Tensor View:
        ###         https://pytorch.org/docs/stable/generated/torch.Tensor.view.html
        ###     Tensor Concatenation:
        ###         https://pytorch.org/docs/stable/generated/torch.cat.html
        ###     Tanh:
        ###         https://pytorch.org/docs/stable/generated/torch.tanh.html
        alpha_t = F.softmax(e_t, dim=1)  # (b, src_len)
        a_t = torch.bmm(alpha_t.unsqueeze(1), enc_hiddens).squeeze(1)  # (b, 2h)
        U_t = torch.cat((a_t, dec_hidden), dim=1)  # (b, 3h), u_t = [a_t; h_t^dec]
        V_t = self.combined_output_projection(U_t)  # (b, h)
        O_t = self.dropout(torch.tanh(V_t))
        ### END YOUR CODE

        combined_output = O_t
        return dec_state, combined_output, e_t

    def generate_sent_masks(self, enc_hiddens: torch.Tensor, source_lengths: List[int]) -> torch.Tensor:
        """ Generate sentence masks for encoder hidden states.

        @param enc_hiddens (Tensor): encodings of shape (b, src_len, 2*h), where b = batch size,
                                     src_len = max source length, h = hidden size.
        @param source_lengths (List[int]): List of actual lengths for each of the sentences in the batch.

        @returns enc_masks (Tensor): Tensor of sentence masks of shape (b, src_len),
                                    where src_len = max source length, h = hidden size.
        """
        enc_masks = torch.zeros(enc_hiddens.size(0), enc_hiddens.size(1), dtype=torch.float)
        for e_id, src_len in enumerate(source_lengths):
            enc_masks[e_id, src_len:] = 1
        return enc_masks.to(self.device)

    def beam_search(self, src_sent: List[str], beam_size: int = 5, max_decoding_time_step: int = 70) -> List[
        Hypothesis]:
        """ Given a single source sentence, perform beam search, yielding translations in the target language.
        @param src_sent (List[str]): a single source sentence (words)
        @param beam_size (int): beam size
        @param max_decoding_time_step (int): maximum number of time steps to unroll the decoding RNN
        @returns hypotheses (List[Hypothesis]): a list of hypothesis, each hypothesis has two fields:
                value: List[str]: the decoded target sentence, represented as a list of words
                score: float: the log-likelihood of the target sentence
        """
        src_sents_var = self.vocab.src.to_input_tensor([src_sent], self.device)

        src_encodings, dec_init_vec = self.encode(src_sents_var, [len(src_sent)])
        src_encodings_att_linear = self.att_projection(src_encodings)

        h_tm1 = dec_init_vec
        att_tm1 = torch.zeros(1, self.hidden_size, device=self.device)

        eos_id = self.vocab.tgt['</s>']

        hypotheses = [['<s>']]
        hyp_scores = torch.zeros(len(hypotheses), dtype=torch.float, device=self.device)
        completed_hypotheses = []

        t = 0
        while len(completed_hypotheses) < beam_size and t < max_decoding_time_step:
            t += 1
            hyp_num = len(hypotheses)

            exp_src_encodings = src_encodings.expand(hyp_num,
                                                     src_encodings.size(1),
                                                     src_encodings.size(2))

            exp_src_encodings_att_linear = src_encodings_att_linear.expand(hyp_num,
                                                                           src_encodings_att_linear.size(1),
                                                                           src_encodings_att_linear.size(2))

            y_tm1 = torch.tensor([self.vocab.tgt[hyp[-1]] for hyp in hypotheses], dtype=torch.long, device=self.device)
            y_t_embed = self.model_embeddings.target(y_tm1)

            x = torch.cat([y_t_embed, att_tm1], dim=-1)

            (h_t, cell_t), att_t, _ = self.step(x, h_tm1,
                                                exp_src_encodings, exp_src_encodings_att_linear, enc_masks=None)

            # log probabilities over target words
            log_p_t = F.log_softmax(self.target_vocab_projection(att_t), dim=-1)

            live_hyp_num = beam_size - len(completed_hypotheses)
            contiuating_hyp_scores = (hyp_scores.unsqueeze(1).expand_as(log_p_t) + log_p_t).view(-1)
            top_cand_hyp_scores, top_cand_hyp_pos = torch.topk(contiuating_hyp_scores, k=live_hyp_num)

            prev_hyp_ids = torch.div(top_cand_hyp_pos, len(self.vocab.tgt), rounding_mode='floor')
            hyp_word_ids = top_cand_hyp_pos % len(self.vocab.tgt)

            new_hypotheses = []
            live_hyp_ids = []
            new_hyp_scores = []

            for prev_hyp_id, hyp_word_id, cand_new_hyp_score in zip(prev_hyp_ids, hyp_word_ids, top_cand_hyp_scores):
                prev_hyp_id = prev_hyp_id.item()
                hyp_word_id = hyp_word_id.item()
                cand_new_hyp_score = cand_new_hyp_score.item()

                hyp_word = self.vocab.tgt.id2word[hyp_word_id]
                new_hyp_sent = hypotheses[prev_hyp_id] + [hyp_word]
                if hyp_word == '</s>':
                    completed_hypotheses.append(Hypothesis(value=new_hyp_sent[1:-1],
                                                           score=cand_new_hyp_score))
                else:
                    new_hypotheses.append(new_hyp_sent)
                    live_hyp_ids.append(prev_hyp_id)
                    new_hyp_scores.append(cand_new_hyp_score)

            if len(completed_hypotheses) == beam_size:
                break

            live_hyp_ids = torch.tensor(live_hyp_ids, dtype=torch.long, device=self.device)
            h_tm1 = (h_t[live_hyp_ids], cell_t[live_hyp_ids])
            att_tm1 = att_t[live_hyp_ids]

            hypotheses = new_hypotheses
            hyp_scores = torch.tensor(new_hyp_scores, dtype=torch.float, device=self.device)

        if len(completed_hypotheses) == 0:
            completed_hypotheses.append(Hypothesis(value=hypotheses[0][1:],
                                                   score=hyp_scores[0].item()))

        completed_hypotheses.sort(key=lambda hyp: hyp.score, reverse=True)

        return completed_hypotheses

    @property
    def device(self) -> torch.device:
        """ Determine which device to place the Tensors upon, CPU or GPU.
        """
        return self.model_embeddings.source.weight.device

    @staticmethod
    def load(model_path: str):
        """ Load the model from a file.
        @param model_path (str): path to model
        """
        # weights_only=False: the checkpoint pickles the Vocab object (torch >= 2.6 defaults to True)
        params = torch.load(model_path, map_location=lambda storage, loc: storage, weights_only=False)
        args = params['args']
        model = NMT(vocab=params['vocab'], **args)
        model.load_state_dict(params['state_dict'])

        return model

    def save(self, path: str):
        """ Save the odel to a file.
        @param path (str): path to the model
        """
        print('save model parameters to [%s]' % path, file=sys.stderr)

        params = {
            'args': dict(embed_size=self.model_embeddings.embed_size, hidden_size=self.hidden_size,
                         dropout_rate=self.dropout_rate),
            'vocab': self.vocab,
            'state_dict': self.state_dict()
        }

        torch.save(params, path)

In [ ]:
%%writefile run.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
CS224N Spring 2024: Assignment 3
run.py: Run Script for Simple NMT Model
Pencheng Yin <pcyin@cs.cmu.edu>
Sahil Chopra <schopra8@stanford.edu>
Vera Lin <veralin@stanford.edu>
Siyan Li <siyanli@stanford.edu>
Moussa KB Doumbouya <moussa@stanford.edu>

Usage:
    run.py train --train-src=<file> --train-tgt=<file> --dev-src=<file> --dev-tgt=<file> --vocab=<file> [options]
    run.py decode [options] MODEL_PATH TEST_SOURCE_FILE OUTPUT_FILE
    run.py decode [options] MODEL_PATH TEST_SOURCE_FILE TEST_TARGET_FILE OUTPUT_FILE

Options:
    -h --help                               show this screen.
    --cuda                                  use GPU
    --train-src=<file>                      train source file
    --train-tgt=<file>                      train target file
    --dev-src=<file>                        dev source file
    --dev-tgt=<file>                        dev target file
    --vocab=<file>                          vocab file
    --seed=<int>                            seed [default: 0]
    --batch-size=<int>                      batch size [default: 32]
    --embed-size=<int>                      embedding size [default: 1024]
    --hidden-size=<int>                     hidden size [default: 768]
    --clip-grad=<float>                     gradient clipping [default: 5.0]
    --log-every=<int>                       log every [default: 10]
    --max-epoch=<int>                       max epoch [default: 30]
    --input-feed                            use input feeding
    --patience=<int>                        wait for how many iterations to decay learning rate [default: 5]
    --max-num-trial=<int>                   terminate training after how many trials [default: 5]
    --lr-decay=<float>                      learning rate decay [default: 0.5]
    --beam-size=<int>                       beam size [default: 10]
    --sample-size=<int>                     sample size [default: 5]
    --lr=<float>                            learning rate [default: 0.001]
    --uniform-init=<float>                  uniformly initialize all parameters [default: 0.1]
    --save-to=<file>                        model save path [default: model.bin]
    --valid-niter=<int>                     perform validation after how many iterations [default: 2000]
    --dropout=<float>                       dropout [default: 0.3]
    --max-decoding-time-step=<int>          maximum number of decoding time steps [default: 70]
"""
import math
import sys
import pickle
import time
from pathlib import Path


from docopt import docopt
from beam_search_diagnostics import format_example_sentence
# from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
import sacrebleu
from nmt_model import Hypothesis, NMT
import numpy as np
from typing import List, Tuple, Dict, Set, Union
from tqdm import tqdm
from utils import read_corpus, batch_iter
from vocab import Vocab, VocabEntry

import torch
import torch.nn.utils
from torch.utils.tensorboard import SummaryWriter


def evaluate_ppl(model, dev_data, batch_size=32):
    """ Evaluate perplexity on dev sentences
    @param model (NMT): NMT Model
    @param dev_data (list of (src_sent, tgt_sent)): list of tuples containing source and target sentence
    @param batch_size (batch size)
    @returns ppl (perplixty on dev sentences)
    """
    was_training = model.training
    model.eval()

    cum_loss = 0.
    cum_tgt_words = 0.

    # no_grad() signals backend to throw away all gradients
    with torch.no_grad():
        for src_sents, tgt_sents in batch_iter(dev_data, batch_size):
            loss = -model(src_sents, tgt_sents).sum()

            cum_loss += loss.item()
            tgt_word_num_to_predict = sum(len(s[1:]) for s in tgt_sents)  # omitting leading `<s>`
            cum_tgt_words += tgt_word_num_to_predict

        ppl = np.exp(cum_loss / cum_tgt_words)

    if was_training:
        model.train()

    return ppl


def compute_corpus_level_bleu_score(references: List[List[str]], hypotheses: List[Hypothesis]) -> float:
    """ Given decoding results and reference sentences, compute corpus-level BLEU score.
    @param references (List[List[str]]): a list of gold-standard reference target sentences
    @param hypotheses (List[Hypothesis]): a list of hypotheses, one for each reference
    @returns bleu_score: corpus-level BLEU score
    """
    # remove the start and end tokens
    if references[0][0] == '<s>':
        references = [ref[1:-1] for ref in references]
    
    # detokenize the subword pieces to get full sentences
    detokened_refs = [''.join(pieces).replace('▁', ' ') for pieces in references]
    detokened_hyps = [''.join(hyp.value).replace('▁', ' ') for hyp in hypotheses]

    # sacreBLEU can take multiple references (golden example per sentence) but we only feed it one
    bleu = sacrebleu.corpus_bleu(detokened_hyps, [detokened_refs])

    return bleu.score


def train(args: Dict):
    """ Train the NMT Model.
    @param args (Dict): args from cmd line
    """
    train_data_src = read_corpus(args['--train-src'], source='src', vocab_size=21000)       # EDIT: NEW VOCAB SIZE
    train_data_tgt = read_corpus(args['--train-tgt'], source='tgt', vocab_size=8000)

    dev_data_src = read_corpus(args['--dev-src'], source='src', vocab_size=3000)
    dev_data_tgt = read_corpus(args['--dev-tgt'], source='tgt', vocab_size=2000)

    example_sentence_src = dev_data_src[3]
    example_sentence_tgt = dev_data_tgt[3]

    train_data = list(zip(train_data_src, train_data_tgt))
    dev_data = list(zip(dev_data_src, dev_data_tgt))

    train_batch_size = int(args['--batch-size'])
    clip_grad = float(args['--clip-grad'])
    valid_niter = int(args['--valid-niter'])
    log_every = int(args['--log-every'])
    model_save_path = args['--save-to']

    vocab = Vocab.load(args['--vocab'])

    # The course setup uses embed_size=1024 / hidden_size=768, which are now the CLI defaults
    # (previously hard-coded here), so they can be shrunk for quick CPU smoke tests.
    model = NMT(embed_size=int(args['--embed-size']),
                hidden_size=int(args['--hidden-size']),
                dropout_rate=float(args['--dropout']),
                vocab=vocab)
    
    tensorboard_path = "nmt" if args['--cuda'] else "nmt_local"
    writer = SummaryWriter(log_dir=f"./runs/{tensorboard_path}")
    model.train()

    uniform_init = float(args['--uniform-init'])
    if np.abs(uniform_init) > 0.:
        print('uniformly initialize parameters [-%f, +%f]' % (uniform_init, uniform_init), file=sys.stderr)
        for p in model.parameters():
            p.data.uniform_(-uniform_init, uniform_init)

    vocab_mask = torch.ones(len(vocab.tgt))
    vocab_mask[vocab.tgt['<pad>']] = 0

    device = torch.device("cuda:0" if args['--cuda'] else "cpu")
    print('use device: %s' % device, file=sys.stderr)

    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=float(args['--lr']))
    # optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)                       # EDIT: SMALLER LEARNING RATE

    num_trial = 0
    train_iter = patience = cum_loss = report_loss = cum_tgt_words = report_tgt_words = 0
    cum_examples = report_examples = epoch = valid_num = 0
    hist_valid_scores = []
    train_time = begin_time = time.time()
    print('begin Maximum Likelihood training')

    while True:
        epoch += 1

        for src_sents, tgt_sents in batch_iter(train_data, batch_size=train_batch_size, shuffle=True):
            train_iter += 1

            optimizer.zero_grad()

            batch_size = len(src_sents)

            example_losses = -model(src_sents, tgt_sents) # (batch_size,)
            batch_loss = example_losses.sum()
            loss = batch_loss / batch_size

            loss.backward()

            # clip gradient
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

            optimizer.step()

            batch_losses_val = batch_loss.item()
            report_loss += batch_losses_val
            cum_loss += batch_losses_val

            tgt_words_num_to_predict = sum(len(s[1:]) for s in tgt_sents)  # omitting leading `<s>`
            report_tgt_words += tgt_words_num_to_predict
            cum_tgt_words += tgt_words_num_to_predict
            report_examples += batch_size
            cum_examples += batch_size

            if train_iter % log_every == 0:
                writer.add_scalar("loss/train", report_loss / report_tgt_words, train_iter)
                writer.add_scalar("perplexity/train", math.exp(report_loss / report_tgt_words), train_iter)
                print('epoch %d, iter %d, avg. loss %.2f, avg. ppl %.2f ' \
                      'cum. examples %d, speed %.2f words/sec, time elapsed %.2f sec' % (epoch, train_iter,
                                                                                         report_loss / report_tgt_words,
                                                                                         math.exp(report_loss / report_tgt_words),
                                                                                         cum_examples,
                                                                                         report_tgt_words / (time.time() - train_time),
                                                                                         time.time() - begin_time), file=sys.stderr)

                train_time = time.time()
                report_loss = report_tgt_words = report_examples = 0.

            # perform validation
            if train_iter % valid_niter == 0:
                writer.add_scalar("loss/val", cum_loss / cum_tgt_words, train_iter)
                print('epoch %d, iter %d, cum. loss %.2f, cum. ppl %.2f cum. examples %d' % (epoch, train_iter,
                                                                                         cum_loss / cum_tgt_words,
                                                                                         np.exp(cum_loss / cum_tgt_words),
                                                                                         cum_examples), file=sys.stderr)

                cum_loss = cum_examples = cum_tgt_words = 0.
                valid_num += 1

                print('begin validation ...', file=sys.stderr)

                # compute dev. ppl and bleu
                dev_ppl = evaluate_ppl(model, dev_data, batch_size=128)   # dev batch size can be a bit larger
                example_hypothesis_beam = beam_search(
                    model, [example_sentence_src],
                    beam_size=10,
                    max_decoding_time_step=int(args['--max-decoding-time-step'])
                )[0]
                
                valid_metric = -dev_ppl

                writer.add_scalar("perplexity/val", dev_ppl, train_iter)

                
                writer.add_text(
                    f'example_translation_with_beam_search',
                    format_example_sentence(example_sentence_src, example_sentence_tgt, example_hypothesis_beam, train_iter), 
                    train_iter
                )

                print('validation: iter %d, dev. ppl %f' % (train_iter, dev_ppl), file=sys.stderr)

                is_better = len(hist_valid_scores) == 0 or valid_metric > max(hist_valid_scores)
                hist_valid_scores.append(valid_metric)

                if is_better:
                    patience = 0
                    print('save currently the best model to [%s]' % model_save_path, file=sys.stderr)
                    model.save(model_save_path)

                    # also save the optimizers' state
                    torch.save(optimizer.state_dict(), model_save_path + '.optim')
                elif patience < int(args['--patience']):
                    patience += 1
                    print('hit patience %d' % patience, file=sys.stderr)

                    if patience == int(args['--patience']):
                        num_trial += 1
                        print('hit #%d trial' % num_trial, file=sys.stderr)
                        if num_trial == int(args['--max-num-trial']):
                            print('early stop!', file=sys.stderr)
                            exit(0)

                        # decay lr, and restore from previously best checkpoint
                        lr = optimizer.param_groups[0]['lr'] * float(args['--lr-decay'])
                        print('load previously best model and decay learning rate to %f' % lr, file=sys.stderr)

                        # load model
                        params = torch.load(model_save_path, map_location=lambda storage, loc: storage,
                                            weights_only=False)  # checkpoint pickles the Vocab object
                        model.load_state_dict(params['state_dict'])
                        model = model.to(device)

                        print('restore parameters of the optimizers', file=sys.stderr)
                        optimizer.load_state_dict(torch.load(model_save_path + '.optim'))

                        # set new lr
                        for param_group in optimizer.param_groups:
                            param_group['lr'] = lr

                        # reset patience
                        patience = 0

                if epoch == int(args['--max-epoch']):
                    print('reached maximum number of epochs!', file=sys.stderr)
                    exit(0)


def decode(args: Dict[str, str]):
    """ Performs decoding on a test set, and save the best-scoring decoding results.
    If the target gold-standard sentences are given, the function also computes
    corpus-level BLEU score.
    @param args (Dict): args from cmd line
    """

    print("load test source sentences from [{}]".format(args['TEST_SOURCE_FILE']), file=sys.stderr)
    test_data_src = read_corpus(args['TEST_SOURCE_FILE'], source='src', vocab_size=3000)
    if args['TEST_TARGET_FILE']:
        print("load test target sentences from [{}]".format(args['TEST_TARGET_FILE']), file=sys.stderr)
        test_data_tgt = read_corpus(args['TEST_TARGET_FILE'], source='tgt', vocab_size=2000)

    print("load model from {}".format(args['MODEL_PATH']), file=sys.stderr)
    model = NMT.load(args['MODEL_PATH'])

    if args['--cuda']:
        model = model.to(torch.device("cuda:0"))

    hypotheses = beam_search(model, test_data_src,
                             beam_size=int(args['--beam-size']),  # defaults to 10 (course setting)
                             max_decoding_time_step=int(args['--max-decoding-time-step']))

    if args['TEST_TARGET_FILE']:
        top_hypotheses = [hyps[0] for hyps in hypotheses]
        bleu_score = compute_corpus_level_bleu_score(test_data_tgt, top_hypotheses)
        print('Corpus BLEU: {}'.format(bleu_score), file=sys.stderr)

    Path(args['OUTPUT_FILE']).parent.mkdir(parents=True, exist_ok=True)

    with open(args['OUTPUT_FILE'], 'w', encoding='utf8') as f:
        for src_sent, hyps in zip(test_data_src, hypotheses):
            top_hyp = hyps[0]
            hyp_sent = ''.join(top_hyp.value).replace('▁', ' ')
            f.write(hyp_sent + '\n')


def beam_search(model: NMT, test_data_src: List[List[str]], beam_size: int, max_decoding_time_step: int) -> List[List[Hypothesis]]:
    """ Run beam search to construct hypotheses for a list of src-language sentences.
    @param model (NMT): NMT Model
    @param test_data_src (List[List[str]]): List of sentences (words) in source language, from test set.
    @param beam_size (int): beam_size (# of hypotheses to hold for a translation at every step)
    @param max_decoding_time_step (int): maximum sentence length that Beam search can produce
    @returns hypotheses (List[List[Hypothesis]]): List of Hypothesis translations for every source sentence.
    """
    was_training = model.training
    model.eval()

    hypotheses = []
    with torch.no_grad():
        for src_sent in tqdm(test_data_src, desc='Decoding', file=sys.stdout):
            example_hyps = model.beam_search(src_sent, beam_size=beam_size, max_decoding_time_step=max_decoding_time_step)

            hypotheses.append(example_hyps)

    if was_training: model.train(was_training)

    return hypotheses


def main():
    """ Main func.
    """
    args = docopt(__doc__)

    # Check pytorch version
    assert(torch.__version__ >= "1.0.0"), "Please update your installation of PyTorch. You have {} and you should have version 1.0.0".format(torch.__version__)

    # seed the random number generators
    seed = int(args['--seed'])
    torch.manual_seed(seed)
    if args['--cuda']:
        torch.cuda.manual_seed(seed)
    np.random.seed(seed * 13 // 7)

    if args['train']:
        train(args)
    elif args['decode']:
        decode(args)
    else:
        raise RuntimeError('invalid run mode')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile beam_search_diagnostics.py
import os
import getpass
from datetime import datetime
import socket
import base64
import json
from pathlib import Path
import os


def get_diagnostic_dir():
    diag_path = Path(os.getcwd()) / "outputs" / "beam_search_diagnostics" 
    # diag_path = Path('/') / "outputs" / "beam_search_diagnostics" 
    diag_path.mkdir(parents=True, exist_ok=True)
    return diag_path

def get_username():
    try:
        return os.getlogin()
    except OSError:  # no controlling terminal, e.g. inside Kaggle/Docker containers
        return getpass.getuser()

def get_diagnostic_info():
    d = {
        "t": datetime.utcnow().isoformat(),
        "h": socket.gethostname(),
        "u": get_username()
    }
    s = base64.b64encode(json.dumps(d).encode("utf-8")).decode("utf-8") 
    return s

def record_train_diagnostics(data, iter):
    file = get_diagnostic_dir() / f"{iter:06}.json"
    file.write_text(data, encoding="utf-8")

def format_example_sentence(source, target, hypothesis_beam, iter):
    hypotheses = [
        {
            "hypothesis": h.value,
            "score":h.score
        } 
        for h in hypothesis_beam
    ]
    
    formatted_json = json.dumps(
        {
            "example_source": source,
            "example_target": target,
            "hypotheses": hypotheses,
            "diagnostic_info": get_diagnostic_info()
        },  
        ensure_ascii=False, 
        indent=4
    )

    record_train_diagnostics(formatted_json, iter)

    return f"""## Example of translation with beam search @ Iteration {iter}:
```
{formatted_json}
```
    """

## 5. Official sanity checks

In [ ]:
for part in ("1d", "1e", "1f"):
    run([sys.executable, "sanity_check.py", part], show=lambda line: "All Sanity Checks Passed" in line)

## 6. Training

This runs the same command as `sh run.sh train`. Every line goes to `logs/train.log`. The cell shows the validation results and every 1000th training-progress line. TensorBoard logs go to `student/runs/`.

In [ ]:
for stale in ("runs", "outputs"):
    shutil.rmtree(CODE_DIR / stale, ignore_errors=True)
for stale in ("model.bin", "model.bin.optim"):
    (CODE_DIR / stale).unlink(missing_ok=True)

PROGRESS = re.compile(r"^epoch \d+, iter (\d+), avg\. loss")
QUIET = ("Decoding", "begin validation", "save model parameters", "save currently", "restore parameters")

def show_train_line(line):
    m = PROGRESS.match(line)
    if m:
        return int(m.group(1)) % SHOW_EVERY == 0
    return not line.startswith(QUIET) and ", cum. loss" not in line

def safe_to_stop(line):  # between validations, so no checkpoint is being written
    m = PROGRESS.match(line)
    return bool(m) and int(m.group(1)) % VALID_NITER != 0

train_cmd = [sys.executable, "run.py", "train",
             f"--train-src={DATA_DIR / 'train.zh'}", f"--train-tgt={DATA_DIR / 'train.en'}",
             f"--dev-src={DATA_DIR / 'dev.zh'}", f"--dev-tgt={DATA_DIR / 'dev.en'}",
             "--vocab=vocab.json", *CUDA_FLAG, *TRAIN_ARGS]
t0 = time.time()
train_log = run(train_cmd, log_path=LOG_DIR / "train.log", show=show_train_line,
                deadline=t0 + TRAIN_TIME_BUDGET_H * 3600, can_stop=safe_to_stop)
train_minutes = (time.time() - t0) / 60

validations = [(int(i), float(p)) for i, p in re.findall(r"validation: iter (\d+), dev\. ppl ([\d.]+)", train_log)]
best_iter, best_ppl = min(validations, key=lambda v: v[1])
last_iter = max(int(i) for i in re.findall(r"iter (\d+)", train_log))
lr_decays, last_val = [], None
for line in train_log.splitlines():
    if line.startswith("validation: iter"):
        last_val = int(line.split()[2].rstrip(","))
    elif line.startswith("load previously best model and decay learning rate"):
        lr_decays.append((last_val, float(line.rsplit(" ", 1)[1])))
stop_reason = ("early stopping" if "early stop!" in train_log else
               "max epochs" if "reached maximum number of epochs" in train_log else "time budget")
print(f"\nstopped by {stop_reason} after {last_iter:,} iterations ({train_minutes:.0f} min); "
      f"best dev perplexity {best_ppl:.2f} at iteration {best_iter:,}; {len(lr_decays)} learning-rate decays")

### Training curves

Vertical lines mark the validations after which the best checkpoint was reloaded and the learning rate halved. A table of the dev-perplexity checkpoints follows the chart.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

SURFACE, INK, INK_2, MUTED, GRID, AXIS, SERIES = (
    "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#2a78d6")

events = EventAccumulator(str(CODE_DIR / "runs" / ("nmt" if USE_CUDA else "nmt_local")), size_guidance={"scalars": 0})
events.Reload()
scalars = lambda tag: ([e.step for e in events.Scalars(tag)], [e.value for e in events.Scalars(tag)])
loss_x, loss_y = scalars("loss/train")
ppl_x, ppl_y = scalars("perplexity/val")
ema, smoothed = None, []
for v in loss_y:  # TensorBoard-style exponential smoothing
    ema = v if ema is None else 0.9 * ema + 0.1 * v
    smoothed.append(ema)

fig, (ax_loss, ax_ppl) = plt.subplots(1, 2, figsize=(12, 4.2), facecolor=SURFACE)
for ax in (ax_loss, ax_ppl):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(AXIS)
    ax.tick_params(which="both", colors=AXIS, labelcolor=INK_2, labelsize=9)
    ax.margins(x=0.05)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1000:g}K" if abs(v) >= 1000 else f"{v:g}"))
    ax.set_xlabel("training iteration", color=INK_2, fontsize=9)
    for it, _ in lr_decays:  # learning-rate halvings (best checkpoint reloaded)
        ax.axvline(it, color=AXIS, linewidth=0.8, zorder=1)

ax_loss.plot(loss_x, loss_y, color=SERIES, linewidth=0.8, alpha=0.25)
ax_loss.plot(loss_x, smoothed, color=SERIES, linewidth=1.6, solid_capstyle="round")
ax_loss.annotate(f"{smoothed[-1]:.2f}", (loss_x[-1], smoothed[-1]), xytext=(5, 0),
                 textcoords="offset points", va="center", color=INK_2, fontsize=9)
ax_loss.set_title("Training loss per target subword (nats, smoothed)", loc="left", color=INK, fontsize=11)

ax_ppl.plot(ppl_x, ppl_y, color=SERIES, linewidth=1.6, solid_capstyle="round", solid_joinstyle="round")
ax_ppl.set_yscale("log")
ax_ppl.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax_ppl.yaxis.set_minor_formatter(FuncFormatter(lambda v, _: f"{v:g}" if str(v)[0] in "25" else ""))
ax_ppl.plot([best_iter], [best_ppl], "o", color=SERIES, markersize=7, markeredgecolor=SURFACE, markeredgewidth=1.5)
ax_ppl.annotate(f"best {best_ppl:.2f} at {best_iter:,}", (best_iter, best_ppl), xytext=(0, 10),
                textcoords="offset points", ha="center", color=INK_2, fontsize=9)
ax_ppl.set_title("Dev perplexity (log scale)", loc="left", color=INK, fontsize=11)
if lr_decays:
    ax_ppl.text(lr_decays[0][0], 1.0, " lr halved ", transform=ax_ppl.get_xaxis_transform(),
                ha="left", va="top", color=MUTED, fontsize=8)
fig.tight_layout()
fig.savefig(WORK_DIR / "training_curves.png", dpi=150, facecolor=SURFACE)
plt.show()

print("iteration  dev ppl")  # table view of the validation checkpoints
step = max(1, len(validations) // 12)
for it, ppl in sorted(set(validations[::step] + [(best_iter, best_ppl), validations[-1]])):
    print(f"{it:>9,}  {ppl:8.2f}{'  <- best' if it == best_iter else ''}")

## 7. Test-set evaluation

In [ ]:
decode_log = run([sys.executable, "run.py", "decode", "model.bin", DATA_DIR / "test.zh", DATA_DIR / "test.en",
                  "outputs/test_outputs.txt", *CUDA_FLAG],
                 log_path=LOG_DIR / "test_decode.log",
                 show=lambda line: not line.startswith("Decoding"))
BLEU = float(re.search(r"Corpus BLEU: ([\d.]+)", decode_log).group(1))
print(f"\nTest corpus BLEU: {BLEU:.2f}   (the assignment expects > 18)")

def read_lines(path):
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

srcs, refs = read_lines(DATA_DIR / "test.zh"), read_lines(DATA_DIR / "test.en")
hyps = read_lines(CODE_DIR / "outputs" / "test_outputs.txt")
for i in range(6):
    print(f"\n[{i}] ZH : {srcs[i].strip()}\n    REF: {refs[i].strip()}\n    NMT: {hyps[i].strip()}")

## 8. Beam-search diagnostics (Q2d)

At every validation, `run.py` decodes `dev[3]` with beam size 10 and stores the beam in `outputs/beam_search_diagnostics/<iteration>.json`. Q2(d)(i) asks for the top hypothesis at iterations 200, 3000 and the last one. Q2(d)(ii) compares the hypotheses in the final beam.

In [ ]:
diag_dir = CODE_DIR / "outputs" / "beam_search_diagnostics"
records = {int(p.stem): json.loads(p.read_text(encoding="utf-8")) for p in diag_dir.glob("*.json")}
iters = sorted(records)
detok = lambda pieces: "".join(pieces).replace("▁", " ").strip()

example = records[iters[-1]]
print("source   :", detok(example["example_source"]))
print("reference:", detok(example["example_target"][1:-1]), "\n")
wanted = (200, 3000, iters[-1]) if not QUICK_RUN else (iters[0], iters[len(iters) // 2], iters[-1])
for target in wanted:
    it = min(iters, key=lambda i: abs(i - target))
    top = records[it]["hypotheses"][0]
    print(f"iteration {it:>6,}: {detok(top['hypothesis'])}   (log-likelihood {top['score']:.2f})")

print(f"\nFull beam at the last validation (iteration {iters[-1]:,}):")
for h in records[iters[-1]]["hypotheses"]:
    print(f"  {h['score']:7.2f}  {detok(h['hypothesis'])}")

## 9. Try the trained model

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Using padding='same'")  # Conv1d with an even kernel
sys.path.insert(0, str(CODE_DIR))
import sentencepiece as spm
from nmt_model import NMT

nmt = NMT.load(str(CODE_DIR / "model.bin")).to("cuda" if USE_CUDA else "cpu").eval()
src_sp = spm.SentencePieceProcessor(model_file=str(CODE_DIR / "src.model"))

def translate(sentence, beam_size=10):
    with torch.no_grad():
        best = nmt.beam_search(src_sp.encode_as_pieces(sentence), beam_size=beam_size, max_decoding_time_step=70)[0]
    return detok(best.value)

for zh in ["我喜欢甜点。",                        # Figure 1 of the handout
           "贼人其后被警方拘捕及被判处盗窃罪名成立。",   # Q2(b) i
           "当局已经宣布今天是国殇日。",                # Q2(b) iii
           "需要有充足和可预测的资源。"]:               # Q2(c)
    print(f"{zh}\n  → {translate(zh)}")

## 10. Package the results

In [ ]:
submission = WORK_DIR / "assignment3.zip"   # same contents as collect_submission.sh
with zipfile.ZipFile(submission, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(CODE_DIR.glob("*.py")):
        zf.write(path, path.name)
    for folder in ("sanity_check_en_es_data", "outputs"):
        for path in sorted((CODE_DIR / folder).rglob("*")):
            if path.is_file():
                zf.write(path, path.relative_to(CODE_DIR).as_posix())

if not KEEP_OPTIMIZER_STATE:
    (CODE_DIR / "model.bin.optim").unlink(missing_ok=True)

results = {
    "test_bleu": round(BLEU, 2),
    "best_dev_perplexity": round(best_ppl, 2),
    "best_dev_iteration": best_iter,
    "iterations": last_iter,
    "lr_decays": len(lr_decays),
    "stop_reason": stop_reason,
    "train_minutes": round(train_minutes, 1),
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "quick_run": QUICK_RUN,
}
(WORK_DIR / "results.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))
print(f"\n{submission.name}: {submission.stat().st_size / 1e6:.1f} MB | "
      f"model.bin: {(CODE_DIR / 'model.bin').stat().st_size / 1e6:.0f} MB")

### Outputs (Output tab / `/kaggle/working`)

| File | Contents |
|---|---|
| `results.json` | test BLEU, best dev perplexity, iterations, training time |
| `student/model.bin` | best checkpoint (load it with `NMT.load`) |
| `student/outputs/test_outputs.txt` | test-set translations |
| `student/outputs/beam_search_diagnostics/` | beam at every validation (Q2d) |
| `assignment3.zip` | Gradescope coding submission (`*.py`, sanity-check data, outputs) |
| `training_curves.png`, `logs/train.log`, `student/runs/` | training curves, full log, TensorBoard events |